# NMF on ModulAir 00678

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00678
data = pd.read_csv('MOD-00678.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:38Z,577612587,2025-12-31T18:59:38Z,MOD-00678,49.5,0.4,7.221,0.804,0.196,0.017,0.037,...,27.638,34.986,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.24
2025-12-31T23:58:38Z,577610602,2025-12-31T18:58:38Z,MOD-00678,49.1,0.3,6.655,0.625,0.113,0.030,0.016,...,25.989,37.115,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.28
2025-12-31T23:57:38Z,577610603,2025-12-31T18:57:38Z,MOD-00678,49.3,0.3,6.896,0.719,0.185,0.018,0.023,...,25.507,36.403,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.50
2025-12-31T23:56:38Z,577610601,2025-12-31T18:56:38Z,MOD-00678,49.6,0.3,7.017,0.791,0.154,0.019,0.026,...,25.968,36.735,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,1.13
2025-12-31T23:55:38Z,577610600,2025-12-31T18:55:38Z,MOD-00678,49.6,0.3,6.525,0.657,0.140,0.024,0.019,...,26.916,35.683,14307.0,14308.0,14309.0,14465.0,14490.0,14540.0,14515.0,0.90


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:38Z,2025-12-31T18:59:38Z,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
2025-12-31T23:58:38Z,2025-12-31T18:58:38Z,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2025-12-31T23:57:38Z,2025-12-31T18:57:38Z,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
2025-12-31T23:56:38Z,2025-12-31T18:56:38Z,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
2025-12-31T23:55:38Z,2025-12-31T18:55:38Z,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:38Z,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
1,2025-12-31T18:58:38Z,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2,2025-12-31T18:57:38Z,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
3,2025-12-31T18:56:38Z,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
4,2025-12-31T18:55:38Z,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:38,762.896,2.403,27.638,34.986,7.221,0.804,0.196,0.017,0.037,0.012
1,2025-12-31 18:58:38,752.810,2.402,25.989,37.115,6.655,0.625,0.113,0.030,0.016,0.020
2,2025-12-31 18:57:38,766.214,2.468,25.507,36.403,6.896,0.719,0.185,0.018,0.023,0.017
3,2025-12-31 18:56:38,760.254,2.467,25.968,36.735,7.017,0.791,0.154,0.019,0.026,0.011
4,2025-12-31 18:55:38,756.560,2.402,26.916,35.683,6.525,0.657,0.140,0.024,0.019,0.008


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-08-22 07:00:00,682.14,14.34,38.87,3.77,3.87,0.48,0.13,0.03,0.03,0.03
1,2025-08-22 08:00:00,668.14,5.27,40.39,2.80,3.90,0.48,0.13,0.03,0.04,0.03
2,2025-08-22 09:00:00,637.41,4.62,42.42,2.29,3.91,0.45,0.12,0.02,0.03,0.02
3,2025-08-22 10:00:00,618.07,6.22,45.14,2.10,4.13,0.43,0.12,0.02,0.03,0.02
4,2025-08-22 11:00:00,622.68,7.00,47.82,2.14,4.85,0.55,0.15,0.03,0.03,0.02
...,...,...,...,...,...,...,...,...,...,...,...
3118,2025-12-31 14:00:00,736.98,23.61,42.09,2.03,9.58,1.29,0.44,0.13,0.20,0.16
3119,2025-12-31 15:00:00,741.31,24.45,41.62,1.88,8.94,1.16,0.38,0.10,0.14,0.10
3120,2025-12-31 16:00:00,760.21,25.09,40.17,2.20,8.79,0.86,0.23,0.04,0.04,0.02
3121,2025-12-31 17:00:00,775.26,26.34,37.12,2.08,8.00,0.78,0.17,0.02,0.03,0.01


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-22 07:00:00,682.14,14.34,38.87,3.77,3.87,0.48,0.13,0.03,0.03,0.03
2025-08-22 08:00:00,668.14,5.27,40.39,2.80,3.90,0.48,0.13,0.03,0.04,0.03
2025-08-22 09:00:00,637.41,4.62,42.42,2.29,3.91,0.45,0.12,0.02,0.03,0.02
2025-08-22 10:00:00,618.07,6.22,45.14,2.10,4.13,0.43,0.12,0.02,0.03,0.02
2025-08-22 11:00:00,622.68,7.00,47.82,2.14,4.85,0.55,0.15,0.03,0.03,0.02


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-22 07:00:00,0.205227,0.383422,0.528412,0.089762,0.063328,0.017248,0.012172,0.011407,0.013274,0.016393
2025-08-22 08:00:00,0.201015,0.140909,0.549076,0.066667,0.063819,0.017248,0.012172,0.011407,0.017699,0.016393
2025-08-22 09:00:00,0.191770,0.123529,0.576672,0.054524,0.063983,0.016170,0.011236,0.007605,0.013274,0.010929
2025-08-22 10:00:00,0.185951,0.166310,0.613649,0.050000,0.067583,0.015451,0.011236,0.007605,0.013274,0.010929
2025-08-22 11:00:00,0.187338,0.187166,0.650082,0.050952,0.079365,0.019763,0.014045,0.011407,0.013274,0.010929


In [11]:
data.to_csv(r'MOD-00678_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-22 07:00:00,0.219952,0.381697,0.527034,0.045680,0.065013,0.012328,0.009664,0.008583,0.017807,0.020301
2025-08-22 08:00:00,0.192565,0.142428,0.553033,0.044414,0.067257,0.017053,0.010848,0.007659,0.016132,0.019904
2025-08-22 09:00:00,0.196459,0.122876,0.575808,0.045619,0.064881,0.014735,0.008248,0.004738,0.012603,0.016671
2025-08-22 10:00:00,0.210825,0.162551,0.606385,0.048296,0.064203,0.013682,0.008207,0.005315,0.014225,0.018515
2025-08-22 11:00:00,0.224687,0.181498,0.638963,0.051344,0.073996,0.017097,0.010476,0.007011,0.016421,0.020742
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.268214,0.624678,0.558800,0.055295,0.138996,0.059276,0.056244,0.057610,0.081946,0.083466
2025-12-31 15:00:00,0.272344,0.646478,0.550934,0.054515,0.132659,0.047307,0.041835,0.040875,0.058590,0.059343
2025-12-31 16:00:00,0.277048,0.663710,0.531846,0.053110,0.134082,0.032303,0.020223,0.013965,0.019736,0.018865


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.046286,0.007556,0.094268,0.007626
1,0.017003,0.012759,0.128491,0.004909
2,0.014793,0.012053,0.136947,0.001725
3,0.019593,0.010692,0.140371,0.002827
4,0.021836,0.013157,0.146666,0.004027
...,...,...,...,...
3060,0.072903,0.027390,0.070828,0.058537
3061,0.076789,0.024652,0.067815,0.039888
3062,0.080932,0.024466,0.063416,0.008571
3063,0.085533,0.020916,0.049759,0.004306


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.358906,8.136090,3.523358,0.388432,0.626368,0.000000,0.000000,0.000000,0.034472,0.016142
Feature 2,0.599564,0.000000,0.000000,0.146730,3.051941,1.154889,0.586067,0.270331,0.092323,0.000000
Feature 3,1.126980,0.011788,3.817208,0.277532,0.137492,0.000000,0.000000,0.000000,0.064617,0.103870
Feature 4,0.000000,0.524463,0.539382,0.056394,0.000000,0.472245,0.686604,0.857670,1.235595,1.280095


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-08-22 07:00:00,0.046286,0.007556,0.094268,0.007626
2025-08-22 08:00:00,0.017003,0.012759,0.128491,0.004909
2025-08-22 09:00:00,0.014793,0.012053,0.136947,0.001725
2025-08-22 10:00:00,0.019593,0.010692,0.140371,0.002827
2025-08-22 11:00:00,0.021836,0.013157,0.146666,0.004027
...,...,...,...,...
2025-12-31 14:00:00,0.072903,0.027390,0.070828,0.058537
2025-12-31 15:00:00,0.076789,0.024652,0.067815,0.039888
2025-12-31 16:00:00,0.080932,0.024466,0.063416,0.008571


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.358906,8.136090,3.523358,0.388432,0.626368,0.000000,0.000000,0.000000,0.034472,0.016142
Factor 2,0.599564,0.000000,0.000000,0.146730,3.051941,1.154889,0.586067,0.270331,0.092323,0.000000
Factor 3,1.126980,0.011788,3.817208,0.277532,0.137492,0.000000,0.000000,0.000000,0.064617,0.103870
Factor 4,0.000000,0.524463,0.539382,0.056394,0.000000,0.472245,0.686604,0.857670,1.235595,1.280095


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.551779,0.070607,0.371851,0.000000,0.005762
no,0.439216,0.083530,0.442664,0.022175,0.012415
no2,0.976621,0.000000,0.001996,0.021893,-0.000510
o3,0.387758,0.000000,0.592581,0.020643,-0.000982
bin0,0.265791,0.651998,0.082297,0.000000,-0.000086
bin1,0.000000,0.892931,0.000000,0.252212,-0.145143
bin2,0.000000,0.625396,0.000000,0.506099,-0.131495
bin3,0.000000,0.321025,0.000000,0.703534,-0.024560
bin4,0.056890,0.076708,0.150424,0.709136,0.006842
bin5,0.026291,0.000000,0.238643,0.725078,0.009987


In [20]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')